<a href="https://colab.research.google.com/github/yerim410/AI/blob/main/%EA%B3%BC%EC%A0%9C1_202412330_%EC%9D%B4%EC%98%88%EB%A6%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 과제 1 — Python 데이터 처리 (2주차)

**AI Application | 스마트모빌리티학과**

## 과제 안내

- 아래 코드 셀의 `___` (밑줄 3개) 부분을 **직접 채우고** 실행하세요. `# TODO` 주석이 힌트입니다.
- 각 문제 바로 아래에 **자동채점 셀**이 있습니다. 문제 셀 실행 → 채점 셀 실행 순서로 진행하세요.
  - `문제 N 통과!`가 출력되면 정답입니다.
  - 오류(AssertionError)가 나면 다시 풀어 보세요. 몇 번이든 재시도 가능합니다.
- **자동채점 셀(assert가 있는 셀)은 절대 수정하지 마세요. 수정 시 0점 처리됩니다.**
- 구성: **기본 문제 1~5** (각 20점) + **심화 문제 6~7 (선택, 가산점)**
- 셀은 반드시 **위에서부터 순서대로** 실행하세요 (데이터 준비 셀부터!).

| 문제 | 주제 | 배점 |
|---|---|---|
| 1 | NumPy 배열 생성과 통계 | 20 |
| 2 | NumPy 불리언 인덱싱 | 20 |
| 3 | Pandas DataFrame 필터링 | 20 |
| 4 | Pandas groupby 집계 | 20 |
| 5 | 결측치 처리 | 20 |
| 6 (심화·선택) | Min-Max 정규화 | +10 |
| 7 (심화·선택) | IQR 이상치 탐지 | +10 |

## 데이터 준비 (수정 금지, 실행만 하세요)

시연 때 사용한 것과 같은 **합성 차량 주행 로그**를 생성합니다.
1분 간격 24시간(1,440개) 기록: 시각(hour), 속도(speed, km/h), 연비(fuel_eff, km/L), 도로유형(road).

In [24]:
# === 데이터 준비 셀: 수정하지 말고 실행만 하세요 ===
import numpy as np
import pandas as pd

np.random.seed(7)
minute = np.arange(0, 24 * 60)
hour = minute // 60
base = 70 - 25 * np.exp(-((hour - 8) ** 2) / 4) - 28 * np.exp(-((hour - 18) ** 2) / 5)
speed = np.clip(base + np.random.normal(0, 8, minute.size), 0, 130).round(1)
fuel_eff = np.clip(16 - 0.0025 * (speed - 65) ** 2 + np.random.normal(0, 0.8, minute.size), 2, None).round(2)
road = np.where(speed >= 70, "highway", "urban")

log = pd.DataFrame({"hour": hour, "speed": speed, "fuel_eff": fuel_eff, "road": road})
# 결측치 주입 (문제 5용): 연비 센서 40개 소실
nan_idx = np.random.choice(log.index, size=40, replace=False)
log.loc[nan_idx, "fuel_eff"] = np.nan

print("데이터 준비 완료:", log.shape)
log.head()

데이터 준비 완료: (1440, 4)


,hour,speed,fuel_eff,road
0,0,83.5,13.00,highway
1,0,66.3,NaN,urban
2,0,70.3,16.27,highway
3,0,73.3,16.46,highway
4,0,63.7,14.96,urban


## 문제 1 (기본, 20점) — NumPy 배열 생성과 통계

`np.arange`로 0부터 100까지 **10 간격**의 배열을 만들고(0, 10, ..., 100),
`speed` 배열의 **평균**과 **최댓값**을 구하세요.

In [25]:
# TODO: np.arange(시작, 끝, 간격) — 100이 포함되어야 하므로 끝값에 주의!
arr = np.arange(0, 110, 10)

# TODO: speed 배열의 평균과 최댓값 (NumPy 함수 사용)
speed_mean = np.mean(speed)
speed_max = np.max(speed)

print("arr:", arr)
print("평균 속도:", round(speed_mean, 1), "km/h | 최고 속도:", speed_max, "km/h")

arr: [  0  10  20  30  40  50  60  70  80  90 100]
평균 속도: 61.5 km/h | 최고 속도: 93.9 km/h


In [26]:
# === 자동채점 셀 (수정 금지) ===
assert isinstance(arr, np.ndarray), "arr는 np.arange로 만든 NumPy 배열이어야 합니다."
assert arr.size == 11 and arr[0] == 0 and arr[-1] == 100 and arr[1] == 10, "arr는 [0, 10, ..., 100] 이어야 합니다. 끝값을 다시 확인하세요."
assert abs(speed_mean - np.mean(speed)) < 1e-6, "speed_mean이 speed의 평균과 다릅니다."
assert abs(speed_max - np.max(speed)) < 1e-6, "speed_max가 speed의 최댓값과 다릅니다."
print("문제 1 통과!")

문제 1 통과!


## 문제 2 (기본, 20점) — NumPy 불리언 인덱싱

제한속도 80km/h를 **초과**한 속도만 추출하고, 과속 샘플의 **개수**를 구하세요.
(시연 노트북의 과속 구간 분석과 같은 패턴입니다.)

In [27]:
# TODO: 80 초과 조건의 불리언 마스크를 만드세요 (비교 연산자 사용)
over_mask = speed > 80

# TODO: 마스크로 과속 속도만 추출하세요 (불리언 인덱싱)
over_speed = speed[over_mask]

# TODO: 과속 샘플 개수 (np.sum 또는 .size 활용)
n_over = np.sum(over_mask)

print("과속 샘플 수:", n_over, "| 과속 시 평균:", over_speed.mean().round(1), "km/h")

과속 샘플 수: 61 | 과속 시 평균: 83.8 km/h


In [28]:
# === 자동채점 셀 (수정 금지) ===
assert over_mask.dtype == bool, "over_mask는 True/False 불리언 배열이어야 합니다."
assert bool(np.all(over_speed > 80)), "over_speed에 80 이하 값이 섞여 있습니다. '초과' 조건인지 확인하세요."
assert over_speed.size == int(np.sum(speed > 80)), "over_speed 추출이 잘못되었습니다."
assert int(n_over) == int(np.sum(speed > 80)), "n_over가 실제 과속 샘플 수와 다릅니다."
print("문제 2 통과!")

문제 2 통과!


## 문제 3 (기본, 20점) — Pandas 필터링

DataFrame `log`에서 다음 두 가지를 추출하세요.

1. `highway`(고속도로) 주행 행만 모은 `highway_df`
2. **시내(urban)이면서 속도 60 초과**인 행만 모은 `urban_fast` (조건 결합 — 괄호 주의!)

In [29]:
# TODO: road 열이 "highway"인 행만 필터링
highway_df = log[log["road"] == "highway"]

# TODO: road가 "urban" 이면서(&) speed가 60 초과인 행 — 각 조건을 괄호로!
urban_fast = log[(log["road"] == "urban") & (log["speed"] > 60)]

print("고속도로 행 수:", len(highway_df))
print("시내 60km/h 초과 행 수:", len(urban_fast))

고속도로 행 수: 371
시내 60km/h 초과 행 수: 475


In [30]:
# === 자동채점 셀 (수정 금지) ===
assert (highway_df["road"] == "highway").all() and len(highway_df) == (log["road"] == "highway").sum(), "highway_df 필터링이 잘못되었습니다."
assert (urban_fast["road"] == "urban").all(), "urban_fast에 urban이 아닌 행이 있습니다."
assert (urban_fast["speed"] > 60).all(), "urban_fast에 60 이하 속도가 있습니다. '초과' 조건인지 확인하세요."
assert len(urban_fast) == len(log[(log["road"] == "urban") & (log["speed"] > 60)]), "urban_fast 행 수가 다릅니다."
print("문제 3 통과!")

문제 3 통과!


## 문제 4 (기본, 20점) — groupby 집계

**시간대(hour)별 평균 속도** `hourly_mean`과 **도로유형(road)별 평균 연비** `road_fuel`을 구하세요.

형태: `df.groupby("기준열")["대상열"].집계함수()`

In [31]:
# TODO: 시간대별 평균 속도 (기준열: hour, 대상열: speed, 집계: mean)
hourly_mean = log.groupby("hour")["speed"].mean()

# TODO: 도로유형별 평균 연비 (기준열: road, 대상열: fuel_eff)
road_fuel = log.groupby("road")["fuel_eff"].mean()

print("시간대별 평균 속도 (앞 5개):")
print(hourly_mean.round(1).head())
print("\n도로유형별 평균 연비:")
print(road_fuel.round(2))

시간대별 평균 속도 (앞 5개):
hour
0    69.7
1    70.8
2    68.8
3    69.8
4    69.5
Name: speed, dtype: float64

도로유형별 평균 연비:
road
highway    15.65
urban      15.57
Name: fuel_eff, dtype: float64


In [32]:
# === 자동채점 셀 (수정 금지) ===
assert len(hourly_mean) == 24, "hourly_mean은 24개(0~23시)여야 합니다. 기준열을 확인하세요."
assert abs(hourly_mean.loc[3] - log[log["hour"] == 3]["speed"].mean()) < 1e-6, "hourly_mean 계산이 잘못되었습니다."
assert set(road_fuel.index) == {"highway", "urban"}, "road_fuel은 도로유형별 결과여야 합니다."
assert abs(road_fuel["urban"] - log[log["road"] == "urban"]["fuel_eff"].mean()) < 1e-6, "road_fuel은 평균(mean)이어야 합니다."
print("문제 4 통과!")

문제 4 통과!


## 문제 5 (기본, 20점) — 결측치 처리

`log`의 `fuel_eff` 열에는 터널 구간에서 소실된 결측치(NaN)가 있습니다.

1. `fuel_eff` 열의 결측치 **개수**를 구하세요.
2. 결측치를 `fuel_eff`의 **평균값으로 대체**한 새 Series `fuel_filled`를 만드세요.

In [33]:
# TODO: fuel_eff 열의 결측치 개수 — isna()와 sum() 조합
n_missing = log["fuel_eff"].isna().sum()

# TODO: 평균값으로 결측치 대체 — fillna() 사용
fuel_filled = log["fuel_eff"].fillna(log["fuel_eff"].mean())

print("결측치 개수:", n_missing)
print("대체 후 결측치:", fuel_filled.isna().sum())

결측치 개수: 40
대체 후 결측치: 0


In [34]:
# === 자동채점 셀 (수정 금지) ===
assert int(n_missing) == 40, "n_missing이 실제 결측치 개수(40)와 다릅니다. isna().sum()을 확인하세요."
assert fuel_filled.isna().sum() == 0, "fuel_filled에 아직 결측치가 남아 있습니다."
assert abs(fuel_filled[log["fuel_eff"].isna()].iloc[0] - log["fuel_eff"].mean()) < 1e-6, "결측치가 '평균값'으로 대체되지 않았습니다."
assert log["fuel_eff"].isna().sum() == 40, "원본 log를 변경하지 마세요. 새 변수 fuel_filled에만 저장해야 합니다."
print("문제 5 통과!")

문제 5 통과!


## 문제 6 (심화·선택, +10점) — Min-Max 정규화 (Normalization)

`speed` 배열을 Min-Max 정규화하여 0~1 범위의 `speed_norm`을 만드세요.

공식: $x_{norm} = \dfrac{x - x_{min}}{x_{max} - x_{min}}$

In [35]:
# TODO: Min-Max 정규화 공식을 한 줄로 (브로드캐스팅 활용, 반복문 금지)
speed_norm = (speed - speed.min()) / (speed.max() - speed.min())

print("정규화 범위:", speed_norm.min(), "~", speed_norm.max())
print("원본 평균:", speed.mean().round(1), "→ 정규화 평균:", speed_norm.mean().round(3))

정규화 범위: 0.0 ~ 1.0
원본 평균: 61.5 → 정규화 평균: 0.584


In [36]:
# === 자동채점 셀 (수정 금지) ===
assert speed_norm.shape == speed.shape, "speed_norm의 shape이 원본과 다릅니다."
assert abs(speed_norm.min() - 0.0) < 1e-9 and abs(speed_norm.max() - 1.0) < 1e-9, "정규화 결과는 최소 0, 최대 1이어야 합니다."
expected = (speed - speed.min()) / (speed.max() - speed.min())
assert bool(np.allclose(speed_norm, expected)), "Min-Max 공식이 잘못 적용되었습니다."
print("문제 6 통과! (심화)")

문제 6 통과! (심화)


## 문제 7 (심화·선택, +10점) — IQR 이상치 탐지 (Outlier Detection)

아래 `speed_gps`에는 GPS 튐으로 생긴 비정상 값이 섞여 있습니다.
IQR 방법으로 이상치 경계를 계산하고, 이상치를 추출하세요.

- 하한 = Q1 - 1.5 × IQR, 상한 = Q3 + 1.5 × IQR (IQR = Q3 - Q1)
- 이상치 = 하한 **미만** 또는(`|`) 상한 **초과**인 값

In [37]:
# 이상치가 섞인 데이터 (수정 금지)
rng = np.random.default_rng(0)
speed_gps = np.concatenate([speed, np.array([250.0, 300.0, 275.0])])

# TODO: 1사분위수(25)와 3사분위수(75) — np.percentile 사용
q1 = np.percentile(speed_gps, 25)
q3 = np.percentile(speed_gps, 75)
iqr = q3 - q1

# TODO: 하한과 상한 (1.5 * iqr 사용)
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

# TODO: 이상치 추출 — 하한 미만 '또는' 상한 초과 (| 연산자, 괄호 주의)
outliers = speed_gps[(speed_gps < lower) | (speed_gps > upper)]

print(f"정상 범위: {lower:.1f} ~ {upper:.1f} km/h")
print("탐지된 이상치:", np.sort(outliers))

정상 범위: 29.2 ~ 94.8 km/h
탐지된 이상치: [ 16.   22.5  23.3  27.3  27.6  27.9  28.2  28.3 250.  275.  300. ]


In [38]:
# === 자동채점 셀 (수정 금지) ===
assert abs(q1 - np.percentile(speed_gps, 25)) < 1e-6 and abs(q3 - np.percentile(speed_gps, 75)) < 1e-6, "q1/q3 계산이 잘못되었습니다 (25, 75 백분위수)."
assert abs(lower - (q1 - 1.5 * iqr)) < 1e-6, "하한(lower) 공식이 잘못되었습니다."
assert {250.0, 300.0, 275.0}.issubset(set(outliers.tolist())), "주입된 GPS 이상치(250/275/300)가 탐지되지 않았습니다."
assert bool(np.all((outliers < lower) | (outliers > upper))), "outliers에 정상 범위 값이 섞여 있습니다."
print("문제 7 통과! (심화)")

문제 7 통과! (심화)


## 제출 방법

1. **Runtime(런타임) → Run all(모두 실행)** 로 위에서부터 전체 셀을 실행하세요.
   - 기본 문제 1~5의 채점 셀에서 모두 `문제 N 통과!`가 출력되어야 합니다.
   - 심화 문제 6~7은 선택입니다. 풀지 않았다면 해당 문제 셀과 채점 셀 실행 시 오류가 나도 감점은 없습니다 (이 경우 6번 문제 셀부터는 실행하지 않고 제출해도 됩니다).
2. 실행 결과가 보이는 상태로 **저장**하세요 (`Ctrl + S`).
3. 파일명을 **`과제1_학번_이름.ipynb`** 으로 변경하세요. (예: `과제1_20241234_홍길동.ipynb`)
4. **본인 GitHub 저장소**에 push하고, **e-Campus 과제 게시판**에도 같은 파일을 제출하세요.
5. 담당 교수에게 **이메일로 알림**을 보내세요 (제목: `[AI응용] 과제1_학번_이름`, 본문에 GitHub 저장소 링크 포함). GitHub 사용이 어려운 경우에는 .ipynb 파일을 이메일에 첨부해 제출해도 인정됩니다 (e-Campus 제출은 공통으로 필요합니다).

- **기한: 다음 주 수업 전날(화요일) 23:59** — 늦은 제출은 하루당 20% 감점
- 배점: 기본 5문제 × 20점 = 100점, 심화 2문제 = 최대 +20점 가산
- **자동채점 셀을 수정하면 0점 처리됩니다.**
- 질문은 Mattermost 채널에 **에러 메시지 전체를 캡처**해서 올려 주세요.